In [1]:
import pandas as pd
import json
from pathlib import Path
from pprint import pprint

from collections import defaultdict

In [2]:
load_fpath = Path("traducatori_2.json")

In [3]:
def load_data(fpath: Path):
    with open(fpath) as json_file:
        traducatori = json.load(json_file)
    
    return traducatori

traducatori = load_data(fpath=load_fpath)

In [4]:
# Check that all entries have identical fields
assert all([traducatori[0].keys() == traducator_entry.keys() for traducator_entry in traducatori])

# Group by field
grouped = defaultdict(list)
for trad_entry in traducatori:
    for k, v in trad_entry.items():
        grouped[k].append(v)

In [5]:
# Have a look at which fields actually contain useful data 
fields = []
for field in grouped.keys():
    keep_field = len([x for x in grouped[field] if x is not None and x != '' and x != []]) != 0
    if keep_field:
        fields.append(field)

fields = sorted(fields)  # Arguably useless, since we're constructing a set immediately after but useful for denu
print(fields)

['curteApel', 'id', 'judet', 'limbiAutorizate', 'nr', 'numar_autorizatie', 'nume', 'result', 'telefon']


In [6]:
# Check that all results are "OK"
assert(all([x == "OK" for x in grouped["result"]]))

fields = set(fields)
fields.discard("result")  # NOTE: In production, you'd want this to be .remove()
fields = sorted(list(fields))
fields

['curteApel',
 'id',
 'judet',
 'limbiAutorizate',
 'nr',
 'numar_autorizatie',
 'nume',
 'telefon']

In [7]:
filtered_dict = {k:v for k, v in grouped.items() if k in fields}
filtered_dict.keys()

dict_keys(['id', 'nume', 'curteApel', 'judet', 'nr', 'telefon', 'numar_autorizatie', 'limbiAutorizate'])

In [8]:
# bis_numbers = ["6907bis", "5955bis", "5954bis", "5952bis", "4955bis", "483bis", "3959bis", "284bis", "2816bis", "2364bis"]

# auth_nr = [int(x) if x not in bis_numbers else int(x[:-3]) * -1 for x in filtered_dict["numar_autorizatie"]]
# auth_nr = sorted(auth_nr)

# auth_nr[-1]

In [25]:
# Dump to Excel file

allowed_nr_aut = [574, 648, 654, 1427, 1452, 1463, 1501, 1524, 1584, 1632, 1727, 2184, 2245, 2367, 2606, 2689, 3237, 3919, 4050, 4260, 4381, 4920, 5464, 5769, 5901, 6058, 7377, 7546, 7734, 7860, 8102, 8177, 8751, 8818, 9313, 9388, 9484, 9703, 9887, 9959, 10012, 10169, 10181, 10202, 10266, 10383, 10887, 10896, 10899, 10917, 11155, 11844, 11855, 11966, 12149, 12337, 12508, 12511, 12517, 12535, 12565, 12799, 12836, 12852, 12960, 13010, 13070, 13182, 13251, 13388, 13402, 13636, 14084, 14224, 14639, 14583, 14708, 14749, 14924, 15009, 15096, 15149, 15281, 15372, 15407, 15422, 15474, 15921, 16289, 16528, 16561, 16572, 16671, 16855, 17067, 17142, 17791, 17931, 17932, 18093, 18287, 18845, 19168, 19189, 19400, 19605, 19837, 19883, 20266, 20648, 20798, 21097, 21121, 21150, 21354, 21529, 21861, 21869, 21901, 22050, 22330, 22635, 23086, 23383, 23472, 23500, 23902, 23946, 23993, 24110, 24243, 24393, 25105, 25357, 25868, 26158, 26224, 26267, 26807, 26908, 27104, 27683, 27741, 27980, 28123, 28232, 28412, 28859, 28992, 29092, 29187, 29348, 29544, 30019, 30535, 31401, 31922, 32041, 32426, 32770, 33316, 33641, 33665, 33874, 34180, 34485, 34530, 34646, 34667, 34959, 35410, 35530, 35771, 36284, 36466, 36520, 36533, 36840, 36856, 37368, 37402, 37435, 37598, 37638, 37777, 37830, 37922, 38082, 38222, 38249, 38310, 38314, 38332, 38396, 38428, 38429, 38436, 38517, 38564, 38592, 38594, 38604, 38608, 38645, 38678, 38690, 38694, 38718, 38723, 38753, 38758, 38781, 38809, 38828, 38847, 38878, 38887, 38898, 38932, 38942, 38948, 38995, 38997, 39004, 39086, 39138, 39141, 39142, 39169, 39241, 39268, 39279, 39336, 39363, 39379, 39408, 39418, 39420, 39424, 39429, 39432, 39434]
allowed_nr_aut = [str(x) for x in allowed_nr_aut]
allowed_nr_aut = set(allowed_nr_aut)

excel_filtered_dict = {k: [] for k in filtered_dict.keys()}
for i in range(len(filtered_dict["numar_autorizatie"])):
    if filtered_dict["numar_autorizatie"][i] in allowed_nr_aut:
        for k in filtered_dict.keys():
            item = filtered_dict[k][i]
            if k == "numar_autorizatie":
                item = int(item)
            excel_filtered_dict[k].append(item)
translations.pragro@gmail.com
df = pd.DataFrame(excel_filtered_dict)
df.to_excel(load_fpath.with_name(f"{load_fpath.stem}_filtered.xlsx"), index=False)

NameError: name 'translations' is not defined

In [10]:
# All keys must contain lists with the same number of elements!
expected_lengths = [len(filtered_dict[k]) for k in filtered_dict.keys()]
assert(all([expected_lengths[0] == x for x in expected_lengths]))

In [59]:
# Construct .json based on Authorisation Number
# Mapping {'id': [...], 'nume': [...], 'curteApel': [...], 'nr': [...], 'numar_autorizatie': [...], 'limbiAutorizate': [...]}
processed_json = {}
for i in range(expected_lengths[0]):
    processed_json[filtered_dict["numar_autorizatie"][i]] = {k:v[i] for k,v in filtered_dict.items() if k != "numar_autorizatie"}

processed_json[f"{39429}"]

{'id': 39439,
 'nume': 'Olteanu Mihai-Cristian',
 'curteApel': 'PITEŞTI',
 'judet': 'ARGES',
 'nr': 6488,
 'telefon': '0799 996 300, mihaiolteanucristian.pfa@gmail.com',
 'limbiAutorizate': 'Engleză'}

In [12]:
# Dump back to .json
save_fpath = load_fpath.with_stem(f"{load_fpath.stem}_filtered")
with open(save_fpath, "w") as fp:
    json.dump(processed_json, fp, indent=4, ensure_ascii=False)
    

# Processing Excel

In [148]:
# Get initial database for .json
clean_excel_fpath = Path('LISTA 17.06.26_clean.xlsx')

# Read Excel and convert to dict
df = pd.read_excel(clean_excel_fpath, dtype=str).fillna("")
translator_data_reduced = df.to_dict(orient="records")

In [149]:
# Some light housekeeping
for i in range(len(translator_data_reduced)):

    for k in ["Număr de telefon", "E-mail"]:
        translator_data_reduced[i][k] = "".join(translator_data_reduced[i][k].split())

    for k in ["Disponibil interpretariat (DI)", "Disponibil interpretariat la instanțe (DII)", "Înregistrat pe lista ambasadă (LA)"]:
        translator_data_reduced[i][k] = (translator_data_reduced[i][k] == "True")

    translator_data_reduced[i]["Alte detalii"] = translator_data_reduced[i]["Alte detalii"].strip()

In [150]:
# Convert from list of dicts to dict of dicts (based on Authorisation Number)
translator_data_reduced_dict = {}
for i in range(len(translator_data_reduced)):
    translator_data_reduced_dict[translator_data_reduced[i]["Număr Autorizație"]] = {k: v for k, v in translator_data_reduced[i].items() if k != "Număr Autorizație"}

translator_data_reduced_dict["39429"]

{'Număr de telefon': '',
 'E-mail': '',
 'Disponibil interpretariat (DI)': False,
 'Disponibil interpretariat la instanțe (DII)': False,
 'Înregistrat pe lista ambasadă (LA)': False,
 'Alte detalii': 'Înregistrat la Ambasada Marii Britanii la București'}

In [151]:
# Dump back to .json
save_fpath_reduced = clean_excel_fpath.with_name("traducatori_reduced.json")
with open(save_fpath_reduced, "w") as fp:
    json.dump(translator_data_reduced_dict, fp, indent=4, ensure_ascii=False)


# Correlate date with the MJ Database

In [152]:
translator_data_reduced_dict["39429"].keys()

dict_keys(['Număr de telefon', 'E-mail', 'Disponibil interpretariat (DI)', 'Disponibil interpretariat la instanțe (DII)', 'Înregistrat pe lista ambasadă (LA)', 'Alte detalii'])

In [153]:
processed_json["39429"].keys()

dict_keys(['id', 'nume', 'curteApel', 'judet', 'nr', 'telefon', 'limbiAutorizate'])

In [154]:
processed_json_keys = ["nume", "curteApel", "limbiAutorizate", "judet"]

# Ensure that we have different keys!
assert(all([x not in translator_data_reduced_dict["39429"].keys() for x in processed_json_keys]))

In [155]:
translator_dict_with_mj = {k:v for k, v in translator_data_reduced_dict.items()}
for auth_no in translator_dict_with_mj.keys():
    for k in processed_json_keys:
        translator_dict_with_mj[auth_no][k] = processed_json[f"{auth_no}"][k]

In [156]:
translator_dict_with_mj["39429"]

{'Număr de telefon': '',
 'E-mail': '',
 'Disponibil interpretariat (DI)': False,
 'Disponibil interpretariat la instanțe (DII)': False,
 'Înregistrat pe lista ambasadă (LA)': False,
 'Alte detalii': 'Înregistrat la Ambasada Marii Britanii la București',
 'nume': 'Olteanu Mihai-Cristian',
 'curteApel': 'PITEŞTI',
 'limbiAutorizate': 'Engleză',
 'judet': 'ARGES'}

# Document view (.docx)

## Mutate data

In [174]:
# Handle document headers
document_view_dict = {k: {} for k in translator_dict_with_mj.keys()}
for k, v in translator_dict_with_mj.items():
    # Merge contact info
    document_view_dict[k]["Contact"] = "\n".join(filter(None, [v["Număr de telefon"], v["E-mail"], v["Alte detalii"]]))

    # Merge jurisdiction
    document_view_dict[k]["Județ/CA"] = "\n".join(filter(None, [(v["judet"] or "").title(), (v["curteApel"] or "").title()]))  # NOTE: Sometimes, we get None's on the fields and we'd like to treat this as empty space instead

    # Merge flags on Interpreting/Embassy
    DI = "DI" if v["Disponibil interpretariat (DI)"] else ""
    DII = "DII" if v["Disponibil interpretariat la instanțe (DII)"] else ""
    LA = "LA" if v["Înregistrat pe lista ambasadă (LA)"] else ""
    document_view_dict[k]["DI/DII/LA"] = "\n".join(filter(None, [DI, DII, LA]))

    # Change of key name
    document_view_dict[k]["Nume"] = v["nume"]

    # Handle languages
    document_view_dict[k]["Limbă/Limbi"] = sorted(v["limbiAutorizate"].split(", "))

    # Also embed the authorisation number to make things easier later-on
    document_view_dict[k]["Nr. aut."] = k


In [175]:
translator_dict_with_mj["39429"]

{'Număr de telefon': '',
 'E-mail': '',
 'Disponibil interpretariat (DI)': False,
 'Disponibil interpretariat la instanțe (DII)': False,
 'Înregistrat pe lista ambasadă (LA)': False,
 'Alte detalii': 'Înregistrat la Ambasada Marii Britanii la București',
 'nume': 'Olteanu Mihai-Cristian',
 'curteApel': 'PITEŞTI',
 'limbiAutorizate': 'Engleză',
 'judet': 'ARGES'}

In [177]:
document_view_dict["39429"]

{'Contact': 'Înregistrat la Ambasada Marii Britanii la București',
 'Județ/CA': 'Arges\nPiteşti',
 'DI/DII/LA': '',
 'Nume': 'Olteanu Mihai-Cristian',
 'Limbă/Limbi': ['Engleză'],
 'Nr. aut.': '39429'}

## Reorganise data

In [235]:
languages = [v["Limbă/Limbi"] for v in document_view_dict.values()]
languages = sorted(list(set([lang for lang_list in languages for lang in lang_list])))

print(languages)

['Albaneză', 'Arabă', 'Armeană', 'Bulgară', 'Catalană', 'Cehă', 'Chineză', 'Croată', 'Daneză', 'Ebraică', 'Ebraică(ivrit)', 'Engleză', 'Finlandeză', 'Franceză', 'Germană', 'Greacă', 'Greacă veche', 'Italiană', 'Japoneză', 'Latină', 'Lituaniană', 'Macedoneană', 'Maghiară', 'Neerlandeză', 'Neogreacă', 'Norvegiană', 'Olandeză', 'Persană', 'Polonă', 'Portugheză', 'Rromani', 'Rusă', 'Slovacă', 'Slovenă', 'Spaniolă', 'Suedeză', 'Sârbo-croată', 'Sârbă', 'Turcă', 'Ucraineană']


In [243]:
special_languages = ["Olandeză", "Neerlandeză", "Sârbă", "Croată", "Sârbo-croată", "Ebraică", "Ebraică(ivrit)", "Greacă", "Neogreacă", "Greacă veche"]

In [237]:
# Reorganise data by languages (NOTE: handle special cases later-on)
document_view_dict_by_lang = {k: [] for k in languages if k not in special_languages}
for lang in document_view_dict_by_lang.keys():
    for v in document_view_dict.values():
        if lang in set(v["Limbă/Limbi"]):
            document_view_dict_by_lang[lang].append(v)

# Sort items in descending order by Authroisation Number
for v in document_view_dict_by_lang.values():
    v.sort(key=lambda k: int(k["Nr. aut."]), reverse=True)

In [238]:
document_view_dict_by_lang["Engleză"][:3]

[{'Contact': 'ioanaalexandramogos@gmail.com',
  'Județ/CA': 'Bucuresti\nBucureşti',
  'DI/DII/LA': '',
  'Nume': 'Mogoş Ioana-Alexandra',
  'Limbă/Limbi': ['Engleză'],
  'Nr. aut.': '39434'},
 {'Contact': 'Înregistrat la Ambasada Marii Britanii la București',
  'Județ/CA': 'Arges\nPiteşti',
  'DI/DII/LA': '',
  'Nume': 'Olteanu Mihai-Cristian',
  'Limbă/Limbi': ['Engleză'],
  'Nr. aut.': '39429'},
 {'Contact': '+35699562669\ncontact@fasttranslations.eu\nMalta',
  'Județ/CA': 'Timis\nTimişoara',
  'DI/DII/LA': '',
  'Nume': 'Jebeleanu Delia',
  'Limbă/Limbi': ['Engleză'],
  'Nr. aut.': '39420'}]

## Handle Special Languages

In [239]:
# TODO
pass

## Dump to a .docx file

In [240]:
language_to_code = {
    'Albaneză': 'SQ', 
    'Arabă': 'AR',
    'Armeană': 'HY',
    'Bulgară': 'BG',
    'Catalană': 'CA',
    'Cehă': 'CZ',
    'Chineză': 'ZH',
    'Sârbo-croată': 'SR/HR/SH',
    'Croată': 'HR',
    'Sârbă': 'SR',
    'Daneză': 'DK',
    'Ebraică': 'HE',
    'Ebraică(ivrit)': 'HE',
    'Engleză': 'EN',
    'Finlandeză': 'FI',
    'Franceză': 'FR',
    'Germană': 'DE',
    'Greacă': 'EL',
    'Neogreacă': 'EL',
    'Greacă veche': 'greaca veche',
    'Italiană': 'IT',
    'Japoneză': 'JA',
    'Latină': 'LA',
    'Lituaniană': 'LT',
    'Macedoneană': 'MK',
    'Maghiară': 'HU',
    'Neerlandeză': 'NL',
    'Olandeză': 'NL',
    'Norvegiană': 'NO',
    'Persană': 'FA',
    'Polonă': 'PL',
    'Portugheză': 'PT',
    'Rromani': 'RMN',
    'Rusă': 'RU',
    'Slovacă': 'SK',
    'Slovenă': 'SL',
    'Spaniolă': 'ES',
    'Suedeză': 'SU',
    'Turcă': 'TR',
    'Ucraineană': 'UA'
}


In [244]:
from docx import Document
from docx.shared import Cm, Pt
from docx.oxml.ns import qn

from docx.enum.table import WD_ALIGN_VERTICAL
from docx.enum.text import WD_ALIGN_PARAGRAPH

docx_document = Document()

# Narrow margins (in cm)
section = docx_document.sections[0]
section.top_margin = Cm(1.27)
section.bottom_margin = Cm(1.27)
section.left_margin = Cm(1.27)
section.right_margin = Cm(1.27)

# Set default font name and size
style = docx_document.styles["Normal"]
style.font.name = "Arial"
style._element.rPr.rFonts.set(qn("w:eastAsia"), "Arial")
style.font.size = Pt(10)

# Fill with data
for lang, translators_list in document_view_dict_by_lang.items():
    # Add subtitle
    docx_document.add_heading(f"{lang} ({language_to_code[lang]})", level=2)

    # Add Table
    table = docx_document.add_table(rows=1+len(translators_list), cols=len(list(translators_list[0].keys())))  # NOTE: 1+ to account for the header too
    table.style = "Table Grid"

    # Add header
    header = table.rows[0].cells
    header[0].text = "Nr. aut."
    header[1].text = "Nume"
    header[2].text = "Limbă/Limbi"
    header[3].text = "Județ/CA"
    header[4].text = "DI/DII/LA"
    header[5].text = "Contact"

    for idx, translator in enumerate(translators_list):
        cells = table.rows[idx + 1].cells  # NOTE: Offset from the header
        
        cells[0].text = translator["Nr. aut."]
        cells[1].text = translator["Nume"]
        cells[2].text = ", ".join([language_to_code[x] for x in translator["Limbă/Limbi"]])
        cells[3].text = translator["Județ/CA"]
        cells[4].text = translator["DI/DII/LA"]
        cells[5].text = translator["Contact"]

    # Centre content in all cells
    for row in table.rows:
        for cell in row.cells:
            cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            for para in cell.paragraphs:
                para.alignment = WD_ALIGN_PARAGRAPH.CENTER

In [245]:
# Save document
docx_save_fpath = clean_excel_fpath.with_name("traducatori_activi.docx")
docx_document.save(docx_save_fpath)